# NVIDIA Earth 2 / FourCastNet Colab Feasibility Check

This notebook tests whether NVIDIA Earth 2 / FourCastNet can be used as a practical AI weather model route for the weather forecasting and Polymarket trading dissertation.

The purpose is not to train a global weather model from scratch. The purpose is to check whether the public NVIDIA Earth 2 / FourCastNet implementation routes can be installed, inspected and potentially used to generate forecast features for a downstream supervised threshold-exceedance model.

The feasibility criteria are:

1. whether the implementation route is publicly accessible;
2. whether the environment can use GPU acceleration;
3. whether Earth2Studio or FourCastNet packages can be installed;
4. whether a minimal example or model workflow can be inspected or run;
5. whether the output could plausibly provide 2 metre temperature or related forecast variables;
6. whether the route is realistic for the dissertation time frame.

In [1]:
import sys
import platform
import subprocess
from datetime import datetime, timezone

print("Python:", sys.version)
print("Platform:", platform.platform())
print("UTC time:", datetime.now(timezone.utc))

def run_command(command):
    print(f"\n$ {command}")
    result = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True,
    )
    print("Return code:", result.returncode)
    if result.stdout:
        print("STDOUT:")
        print(result.stdout[:3000])
    if result.stderr:
        print("STDERR:")
        print(result.stderr[:3000])
    return result

gpu_check = run_command("nvidia-smi")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
UTC time: 2026-06-16 22:11:17.704657+00:00

$ nvidia-smi
Return code: 0
STDOUT:
Tue Jun 16 22:11:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |   

In [2]:
try:
    import torch
    print("Torch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA device:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch check failed:", repr(e))

Torch version: 2.11.0+cu128
CUDA available: True
CUDA device: Tesla T4


In [3]:
base_install = run_command("pip install -q earth2studio")


$ pip install -q earth2studio
Return code: 0
STDOUT:
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.9/821.9 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.7/77.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.8/17.8 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 102.0 MB/s eta 0:00

In [4]:
try:
    import earth2studio
    print("Earth2Studio imported successfully.")
    print("Earth2Studio version:", getattr(earth2studio, "__version__", "version_not_found"))
except Exception as e:
    print("Earth2Studio import failed:")
    print(repr(e))

Earth2Studio imported successfully.
Earth2Studio version: 0.15.0


In [5]:
import pkgutil

try:
    import earth2studio
    print("Top-level Earth2Studio modules:")
    for module in pkgutil.iter_modules(earth2studio.__path__):
        print("-", module.name)
except Exception as e:
    print("Package inspection failed:", repr(e))

Top-level Earth2Studio modules:
- data
- io
- lexicon
- perturbation
- run
- serve
- statistics
- utils


In [6]:
run_command("rm -rf earth2studio_repo")
clone_e2s = run_command("git clone --depth 1 https://github.com/NVIDIA/earth2studio.git earth2studio_repo")


$ rm -rf earth2studio_repo
Return code: 0

$ git clone --depth 1 https://github.com/NVIDIA/earth2studio.git earth2studio_repo
Return code: 0
STDERR:
Cloning into 'earth2studio_repo'...



In [7]:
run_command("find earth2studio_repo -maxdepth 2 -type f | head -80")
run_command("find earth2studio_repo -maxdepth 3 -iname '*four*' -o -iname '*fcn*' | head -80")


$ find earth2studio_repo -maxdepth 2 -type f | head -80
Return code: 0
STDOUT:
earth2studio_repo/uv.lock
earth2studio_repo/sonar-project.properties
earth2studio_repo/recipes/README.md
earth2studio_repo/tox-smoke.ini
earth2studio_repo/Makefile
earth2studio_repo/.pre-commit-config.yaml
earth2studio_repo/.claude-plugin/marketplace.json
earth2studio_repo/docs/index.md
earth2studio_repo/docs/Makefile
earth2studio_repo/docs/make.bat
earth2studio_repo/docs/sphinxext.py
earth2studio_repo/docs/conf.py
earth2studio_repo/docs/sg_execution_times.rst
earth2studio_repo/docs/readme_graphics.py
earth2studio_repo/CONTRIBUTING.md
earth2studio_repo/pyproject.toml
earth2studio_repo/.cursor-plugin/plugin.json
earth2studio_repo/.gitlab-ci.yml
earth2studio_repo/setup.py
earth2studio_repo/test/conftest.py
earth2studio_repo/test/Dockerfile
earth2studio_repo/earth2studio/run.py
earth2studio_repo/earth2studio/__init__.py
earth2studio_repo/.github/PULL_REQUEST_TEMPLATE.md
earth2studio_repo/CHANGELOG.md
earth2stu

CompletedProcess(args="find earth2studio_repo -maxdepth 3 -iname '*four*' -o -iname '*fcn*' | head -80", returncode=0, stdout='', stderr='')

In [8]:
run_command("find earth2studio_repo -maxdepth 3 -type d | head -80")
run_command("find earth2studio_repo -maxdepth 4 -type f | grep -Ei 'example|fcn|fourcast|forecast|prognostic' | head -120")


$ find earth2studio_repo -maxdepth 3 -type d | head -80
Return code: 0
STDOUT:
earth2studio_repo
earth2studio_repo/recipes
earth2studio_repo/recipes/s2s
earth2studio_repo/recipes/s2s/cfg
earth2studio_repo/recipes/s2s/test
earth2studio_repo/recipes/s2s/src
earth2studio_repo/recipes/eval
earth2studio_repo/recipes/eval/cfg
earth2studio_repo/recipes/eval/docs
earth2studio_repo/recipes/eval/test
earth2studio_repo/recipes/eval/src
earth2studio_repo/recipes/template
earth2studio_repo/recipes/template/cfg
earth2studio_repo/recipes/template/test
earth2studio_repo/recipes/template/src
earth2studio_repo/recipes/hens
earth2studio_repo/recipes/hens/cfg
earth2studio_repo/recipes/hens/test
earth2studio_repo/recipes/hens/src
earth2studio_repo/recipes/tc_tracking
earth2studio_repo/recipes/tc_tracking/cfg
earth2studio_repo/recipes/tc_tracking/test
earth2studio_repo/recipes/tc_tracking/aux_data
earth2studio_repo/recipes/tc_tracking/src
earth2studio_repo/.claude-plugin
earth2studio_repo/docs
earth2studio

CompletedProcess(args="find earth2studio_repo -maxdepth 4 -type f | grep -Ei 'example|fcn|fourcast|forecast|prognostic' | head -120", returncode=0, stdout='earth2studio_repo/docs/userguide/components/prognostic.md\nearth2studio_repo/docs/templates/prognostic.rst\nearth2studio_repo/docs/modules/datasources_forecast.rst\nearth2studio_repo/skills/earth2studio-deterministic-forecast/skill.oms.sig\nearth2studio_repo/skills/earth2studio-deterministic-forecast/skill-card.md\nearth2studio_repo/skills/earth2studio-deterministic-forecast/references/troubleshooting.md\nearth2studio_repo/skills/earth2studio-deterministic-forecast/evals/evals.json\nearth2studio_repo/skills/earth2studio-deterministic-forecast/evals/config.yml\nearth2studio_repo/skills/earth2studio-deterministic-forecast/BENCHMARK.md\nearth2studio_repo/skills/earth2studio-deterministic-forecast/SKILL.md\nearth2studio_repo/skills/earth2studio-create-prognostic/skill.oms.sig\nearth2studio_repo/skills/earth2studio-create-prognostic/skil

In [9]:
fcn_install = run_command('pip install -q "earth2studio[fcn]"')


$ pip install -q "earth2studio[fcn]"
Return code: 0
STDOUT:
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.7/138.7 MB 7.9 MB/s eta 0:00:00

STDERR:
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.6.0 which is incompatible.



In [10]:
try:
    import earth2studio
    print("Earth2Studio still imports after FourCastNet optional install.")
    print("Version:", getattr(earth2studio, "__version__", "version_not_found"))
except Exception as e:
    print("Import failed after FourCastNet optional install:")
    print(repr(e))

Earth2Studio still imports after FourCastNet optional install.
Version: 0.15.0


In [11]:
run_command("rm -rf FourCastNet")
clone_fcn = run_command("git clone --depth 1 https://github.com/NVlabs/FourCastNet.git FourCastNet")


$ rm -rf FourCastNet
Return code: 0

$ git clone --depth 1 https://github.com/NVlabs/FourCastNet.git FourCastNet
Return code: 0
STDERR:
Cloning into 'FourCastNet'...



In [12]:
run_command("find FourCastNet -maxdepth 2 -type f | head -100")
run_command("find FourCastNet -maxdepth 3 -type f | grep -Ei 'infer|predict|forecast|model|README|requirements' | head -120")


$ find FourCastNet -maxdepth 2 -type f | head -100
Return code: 0
STDOUT:
FourCastNet/copernicus/get_data_sfc_short_length.py
FourCastNet/copernicus/get_data_pl_short_length.py
FourCastNet/copernicus/get_data_u_v_6hr.py
FourCastNet/inference/inference.py
FourCastNet/inference/inference_precip.py
FourCastNet/inference/inference_ensemble_precip.py
FourCastNet/inference/inference_ensemble.py
FourCastNet/inference/README_inference.md
FourCastNet/assets/FourCastNet.gif
FourCastNet/assets/nvidia.png
FourCastNet/assets/nersc.png
FourCastNet/export_DDP_vars.sh
FourCastNet/train.py
FourCastNet/utils/logging_utils.py
FourCastNet/utils/img_utils.py
FourCastNet/utils/darcy_loss.py
FourCastNet/utils/data_loader_multifiles.py
FourCastNet/utils/YParams.py
FourCastNet/utils/weighted_acc_rmse.py
FourCastNet/utils/date_time_to_hours.py
FourCastNet/submit_batch.sh
FourCastNet/networks/afnonet.py
FourCastNet/README.md
FourCastNet/.git/HEAD
FourCastNet/.git/description
FourCastNet/.git/index
FourCastNet/.

CompletedProcess(args="find FourCastNet -maxdepth 3 -type f | grep -Ei 'infer|predict|forecast|model|README|requirements' | head -120", returncode=0, stdout='FourCastNet/inference/inference.py\nFourCastNet/inference/inference_precip.py\nFourCastNet/inference/inference_ensemble_precip.py\nFourCastNet/inference/inference_ensemble.py\nFourCastNet/inference/README_inference.md\nFourCastNet/README.md\n', stderr='')

In [13]:
run_command("rm -rf fourcastnet_tutorial")
clone_tutorial = run_command("git clone --depth 1 https://github.com/climatechange-ai-tutorials/fourcastnet.git fourcastnet_tutorial")


$ rm -rf fourcastnet_tutorial
Return code: 0

$ git clone --depth 1 https://github.com/climatechange-ai-tutorials/fourcastnet.git fourcastnet_tutorial
Return code: 0
STDERR:
Cloning into 'fourcastnet_tutorial'...



In [14]:
run_command("find fourcastnet_tutorial -maxdepth 2 -type f | head -100")


$ find fourcastnet_tutorial -maxdepth 2 -type f | head -100
Return code: 0
STDOUT:
fourcastnet_tutorial/README.md
fourcastnet_tutorial/.git/HEAD
fourcastnet_tutorial/.git/description
fourcastnet_tutorial/.git/index
fourcastnet_tutorial/.git/shallow
fourcastnet_tutorial/.git/config
fourcastnet_tutorial/.git/packed-refs
fourcastnet_tutorial/LICENSE
fourcastnet_tutorial/FourCastNet_A_practical_introduction_to_a_state_of_the_art_deep_learning_global_weather_emulator.ipynb



CompletedProcess(args='find fourcastnet_tutorial -maxdepth 2 -type f | head -100', returncode=0, stdout='fourcastnet_tutorial/README.md\nfourcastnet_tutorial/.git/HEAD\nfourcastnet_tutorial/.git/description\nfourcastnet_tutorial/.git/index\nfourcastnet_tutorial/.git/shallow\nfourcastnet_tutorial/.git/config\nfourcastnet_tutorial/.git/packed-refs\nfourcastnet_tutorial/LICENSE\nfourcastnet_tutorial/FourCastNet_A_practical_introduction_to_a_state_of_the_art_deep_learning_global_weather_emulator.ipynb\n', stderr='')

In [15]:
import pandas as pd

rows = []

rows.append({
    "route": "Earth2Studio base package",
    "evidence": "pip install earth2studio and import check",
    "status": "to_fill_after_run",
    "project_use": "Core NVIDIA Earth 2 toolkit route"
})

rows.append({
    "route": "Earth2Studio FourCastNet optional dependency",
    "evidence": "pip install earth2studio[fcn]",
    "status": "to_fill_after_run",
    "project_use": "Potential official FourCastNet route"
})

rows.append({
    "route": "Earth2Studio FourCastNet 3",
    "evidence": "official install route exists but may require heavier CUDA dependencies",
    "status": "not_attempted_initially",
    "project_use": "High value if feasible, but not required for first feasibility pass"
})

rows.append({
    "route": "Original NVlabs FourCastNet repository",
    "evidence": "repository clone and code inspection",
    "status": "to_fill_after_run",
    "project_use": "Original implementation route and literature support"
})

rows.append({
    "route": "Public FourCastNet tutorial",
    "evidence": "tutorial repository clone and notebook inspection",
    "status": "to_fill_after_run",
    "project_use": "Practical Colab demonstration route if official route is too heavy"
})

feasibility_df = pd.DataFrame(rows)
feasibility_df

,route,evidence,status,project_use
0,Earth2Studio base package,pip install earth2studio and import check,to_fill_after_run,Core NVIDIA Earth 2 toolkit route
1,Earth2Studio FourCastNet optional dependency,pip install earth2studio[fcn],to_fill_after_run,Potential official FourCastNet route
2,Earth2Studio FourCastNet 3,official install route exists but may require ...,not_attempted_initially,"High value if feasible, but not required for f..."
3,Original NVlabs FourCastNet repository,repository clone and code inspection,to_fill_after_run,Original implementation route and literature s...
4,Public FourCastNet tutorial,tutorial repository clone and notebook inspection,to_fill_after_run,Practical Colab demonstration route if officia...


In [17]:
feasibility_df.loc[feasibility_df["route"] == "Earth2Studio base package", "status"] = "successful"
feasibility_df.loc[feasibility_df["route"] == "Earth2Studio FourCastNet optional dependency", "status"] = "successful with dependency warning"
feasibility_df.loc[feasibility_df["route"] == "Earth2Studio FourCastNet 3", "status"] = "not attempted in first feasibility pass"
feasibility_df.loc[feasibility_df["route"] == "Original NVlabs FourCastNet repository", "status"] = "repository accessible"
feasibility_df.loc[feasibility_df["route"] == "Public FourCastNet tutorial", "status"] = "repository accessible"

feasibility_df

,route,evidence,status,project_use
0,Earth2Studio base package,pip install earth2studio and import check,successful,Core NVIDIA Earth 2 toolkit route
1,Earth2Studio FourCastNet optional dependency,pip install earth2studio[fcn],successful with dependency warning,Potential official FourCastNet route
2,Earth2Studio FourCastNet 3,official install route exists but may require ...,not attempted in first feasibility pass,"High value if feasible, but not required for f..."
3,Original NVlabs FourCastNet repository,repository clone and code inspection,repository accessible,Original implementation route and literature s...
4,Public FourCastNet tutorial,tutorial repository clone and notebook inspection,repository accessible,Practical Colab demonstration route if officia...


## Feasibility conclusion

The NVIDIA Earth 2 / FourCastNet route is feasible at the Colab environment and implementation-access level. The Colab runtime provides a Tesla T4 GPU, PyTorch recognises CUDA, Earth2Studio installs successfully, and the Earth2Studio package imports correctly. The FourCastNet optional Earth2Studio dependency also installs, although with a dependency warning that should be monitored in later runs.

The original NVlabs FourCastNet repository and a public FourCastNet tutorial repository are both accessible. This confirms that there are practical public routes for further implementation work.

This notebook does not yet demonstrate full FourCastNet inference. Model weights, input data preparation, forecast generation and 2 metre temperature extraction remain unresolved. Therefore, the result should be interpreted as a positive first feasibility check rather than a completed model implementation.

The current conclusion is that NVIDIA Earth 2 / FourCastNet remains a credible provisional second AI weather model route. AIFS / AIFS ENS remains the stronger immediate empirical route because 2 metre temperature forecast-field retrieval has already been tested directly, while NVIDIA Earth 2 / FourCastNet now has a plausible Colab based implementation route for further testing.

## Interpretation of dependency warning

The FourCastNet optional dependency installation completed successfully, but Colab reported a dependency conflict involving `fsspec`. This does not invalidate the feasibility check, because Earth2Studio still imports after installation. However, it may affect later workflows involving datasets or remote file access. Future inference tests should therefore use a clean Colab runtime and record the exact package versions.